# Movie Review Classification

The aim of this notebook is to classify a dataset of film reviews. The dataset contains two columns: review and sentiment. Our dataset is balanced, it contains 498 'negative' reviews and 502 'postive' reviews.

In [32]:
# import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load dataset

We have managed the loading of the data by separating the two columns by the last comma of each row, since the first (review) can contain multiple commas. In addition, it is possible that some lines are empty (the person did not write a review and just shared his feeling), so we ignore them.



In [33]:

rows = []

with open("reviews.csv", encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n").strip()
        
        # Ignore empty lines
        if not line:
            continue
        
        # Check that there is at least one comma
        if "," not in line:
            continue
        
        try:
            review, sentiment = line.rsplit(",", 1)
            rows.append([review, sentiment])
        except ValueError:
            # abnormal line → we skip it
            continue

df = pd.DataFrame(rows, columns=["review", "sentiment"])
df.head()


,review,sentiment
0,review,sentiment
1,One of the other reviewers has mentioned that ...,positive
2,"""A wonderful little production. <br /><br />Th...","positive"""
3,"""I thought this was a wonderful way to spend t...","positive"""
4,Basically there's a family where a little boy ...,negative


## Cleaning data 

We cleaned the text by removing the uppercase, the other characters than the letters, and stop words.

In [34]:
import re
import nltk 
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/bilalmay/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [44]:
corpus = []
for i in range(0, 1000):
  review = re.sub('[^a-zA-Z]', ' ', df['review'][i]) 
  review = review.lower()
  review = review.split()
  ps = PorterStemmer()
  all_stopwords = stopwords.words('english')
  all_stopwords.remove('not')
  review = [ps.stem(word) for word in review if not word in set(all_stopwords)]
  review = ' '.join(review)
  corpus.append(review)

## Tokenization

I vectorize the data with the "bag of words" method, this method will take into account the frequency of each word.

In [45]:
from sklearn.feature_extraction.text import CountVectorizer

In [46]:
cv = CountVectorizer(max_features = 1500)  
X = cv.fit_transform(corpus).toarray()
y = df.iloc[:, -1].values 


## Data separation in training and testing set

I separate the data in 85% train and 15% test.

In [47]:
from sklearn.model_selection import train_test_split

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.15, random_state = 0)

## Model training

I'm  going to train a model of Naive Bayes with our training data.

In [49]:
from sklearn.naive_bayes import GaussianNB

In [50]:
classifier = GaussianNB()
classifier.fit(X_train, y_train)

GaussianNB()

## Prediction on test dataset

In [51]:
y_pred = classifier.predict(X_test)

## Evaluation

The model will be evaluated with a confusion matrix.

In [52]:
from sklearn.metrics import confusion_matrix, accuracy_score

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Gaussian Naïve Bayes model accuracy : {accuracy*100:.2f}%")

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix :")
print(cm)

Accuracy du modèle Naïve Bayes : 35.33%

Matrice de Confusion :
[[18  9  5  3]
 [11 14  3  6]
 [13  2 10 13]
 [12 11  9 11]]


With the model Naive Bayes, we obtain an accuracy score of about 35% which is very low.The model predicts very poorly the classes of our test data. Under 50%, this means that our model predicts at random.

## Other model :

### With TF-IDF  vectorization and Complement Naive Bayes 

To improve our accuracy, I will try to use the complement Naive Bayes model instead of the Gaussian as it is more suitable for discrete data.

In [57]:
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import confusion_matrix, accuracy_score

model = ComplementNB()
model.fit(X_train, y_train)

# prediction
y_pred = classifier.predict(X_test)


In [58]:
from sklearn.metrics import confusion_matrix, accuracy_score

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Complement Naïve Bayes accuracy score : {accuracy*100:.2f}%")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix :")
print(cm)

Accuracy du modèle Naïve Bayes : 42.00%

Matrice de Confusion :
[[15 13  3  4]
 [ 9 18  0  7]
 [ 5  3 12 18]
 [ 5  7 13 18]]


The score is improved by 10%, however the model continues to mispredict classes.


It is also possible that reviews between different movies are too complex as input to predict satisfaction. It is possible that we have overcleaned our data.